# Reranking Over Hybrid Search [Step 3 - Three Retrievers, One Ranking]

> **MLCourse - Agentic AI - Advanced RAG - Reranking**

Module [`../01_hybrid_search`](../01_hybrid_search/README.md) built a retriever
that fuses BM25 and dense search with Reciprocal Rank Fusion. That is a strong
**first stage** - and a first stage is exactly what a cross-encoder needs.

This notebook wires the two together. We deliberately **reuse the RRF retriever
approach from module 01** rather than inventing a new one, so that the only new
component is the reranker, and any difference you see is attributable to it.

The resulting stack - **BM25 + dense -> RRF -> cross-encoder** - is close to
what a well-built production RAG system looks like today.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# Paragraph-sized chunks: human-readable units, good enough for retrieval demos
# and identical to the chunking used in ../01_hybrid_search.
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("characters :", len(raw_text))
print("paragraphs :", len(paragraphs))
print("example    :", paragraphs[10][:150], "...")

characters : 144696
paragraphs : 237
example    : Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passa ...


### 2. Rebuilding the module-01 retriever

The three functions below are the module-01 pipeline in compact form: BM25
ranking, dense ranking, and RRF fusion with `k=60`. If any of this is unfamiliar,
work through
[`../01_hybrid_search/03_reciprocal_rank_fusion.ipynb`](../01_hybrid_search/03_reciprocal_rank_fusion.ipynb)
first - we are not re-explaining RRF here, only using it.

Note the important detail: `hybrid_rank` returns **50** candidates, not 5. As a
first stage its job is recall, and the reranker will do the trimming.

In [4]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([tokenize(p) for p in paragraphs])

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)

print("BM25 documents :", len(paragraphs))
print("dense vectors  :", doc_vectors.shape)


def bm25_rank(query, top_n=50):
    """Return document indices ranked by BM25 score (best first)."""
    scores = bm25.get_scores(tokenize(query))
    return list(np.argsort(scores)[::-1][:top_n])


def dense_rank(query, top_n=50):
    """Return document indices ranked by cosine similarity (best first)."""
    qv = encoder.encode([query], normalize_embeddings=True)[0]
    sims = doc_vectors @ qv
    return list(np.argsort(sims)[::-1][:top_n])


def rrf(rankings, k=60, top_n=50):
    """Reciprocal Rank Fusion - the exact algorithm taught in
    ../01_hybrid_search/03_reciprocal_rank_fusion.ipynb."""
    scores = {}
    for ranked in rankings:
        for rank, doc_id in enumerate(ranked, 1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [int(doc_id) for doc_id, _ in ordered[:top_n]]


def hybrid_rank(query, top_n=50):
    """BM25 + dense, fused with RRF. This is our first-stage retriever."""
    return rrf([bm25_rank(query, top_n), dense_rank(query, top_n)], top_n=top_n)


print("hybrid top-3 for 'the queen and the croquet game':")
for i, doc_id in enumerate(hybrid_rank("the queen and the croquet game", 3), 1):
    print(f"  {i}. doc_{doc_id}: {paragraphs[doc_id][:90]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BM25 documents : 237
dense vectors  : (237, 384)
hybrid top-3 for 'the queen and the croquet game':
  1. doc_150: “Get to your places!” shouted the Queen in a voice of thunder, and people began running ab...
  2. doc_105: The Fish-Footman began by producing from under his arm a great letter, nearly as large as ...
  3. doc_152: The players all played at once without waiting for turns, quarrelling all the while, and f...


### 3. Where RRF stops and reranking starts

RRF is a **positional** algorithm. Its input is nothing but rank numbers:

```
RRF(d) = sum over lists of  1 / (k + rank_of_d_in_that_list)
```

It never looks at a single character of the document. That is a feature - it is
why RRF works with any retriever, needs no training, and cannot be fooled by
score-scale differences between BM25 and cosine similarity.

But it is also a hard ceiling. RRF can only reward *agreement between
retrievers*. If BM25 and the embedding model both like the same off-target
paragraph - which happens constantly, because they share the same lexical and
topical signals - RRF will confidently promote it.

The cross-encoder is the first component in the whole stack that actually
**reads the text against the question**. That is the new information it adds.

In [5]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def hybrid_then_rerank(query, n_candidates=50, top_k=5):
    """The full two-stage pipeline: BM25 + dense -> RRF -> cross-encoder."""
    candidates = hybrid_rank(query, top_n=n_candidates)
    scores = cross_encoder.predict(
        [(query, paragraphs[i]) for i in candidates], batch_size=32
    )
    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return candidates, [(int(i), float(s)) for i, s in ranked[:top_k]]


print("pipeline ready: hybrid_rank (50) -> cross-encoder -> top 5")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

pipeline ready: hybrid_rank (50) -> cross-encoder -> top 5


### 4. Side by side on one query

Let us look at what actually moves. The `stage-1 rank` column shows where each
final result was sitting after RRF.

In [6]:
QUERY = "What happens at the trial of the Knave of Hearts?"

candidates, reranked = hybrid_then_rerank(QUERY, n_candidates=50, top_k=5)

print("QUERY:", QUERY)
print("\n--- STAGE 1: hybrid RRF top 5 ---")
for rank, doc_id in enumerate(candidates[:5], 1):
    print(f"  #{rank} doc_{doc_id}: {paragraphs[doc_id][:110]}...")

print("\n--- STAGE 2: after cross-encoder rerank of 50 candidates ---")
for rank, (doc_id, score) in enumerate(reranked, 1):
    old = candidates.index(doc_id) + 1
    arrow = "NEW" if old > 5 else f"was #{old}"
    print(f"  #{rank} doc_{doc_id} ({arrow}, score={score:+.2f}): "
          f"{paragraphs[doc_id][:110]}...")

QUERY: What happens at the trial of the Knave of Hearts?

--- STAGE 1: hybrid RRF top 5 ---
  #1 doc_200: The King and Queen of Hearts were seated on their throne when they arrived, with a great crowd assembled about...
  #2 doc_143: First came ten soldiers carrying clubs; these were all shaped like the three gardeners, oblong and flat, with ...
  #3 doc_199: “What trial is it?” Alice panted as she ran; but the Gryphon only answered “Come on!” and ran the faster, whil...
  #4 doc_226: “If there’s no meaning in it,” said the King, “that saves a world of trouble, you know, as we needn’t try to f...
  #5 doc_213: “I’m glad I’ve seen that done,” thought Alice. “I’ve so often read in the newspapers, at the end of trials, “T...

--- STAGE 2: after cross-encoder rerank of 50 candidates ---
  #1 doc_200 (was #1, score=+3.60): The King and Queen of Hearts were seated on their throne when they arrived, with a great crowd assembled about...
  #2 doc_143 (was #2, score=-4.08): First came ten soldi

### 5. How much does the ordering actually change?

A useful, cheap diagnostic: how many of the final top-5 were already in the
stage-1 top-5, and how far did documents travel? If the reranker never changes
anything, you are paying latency for nothing - and that is worth knowing.

We run it across several queries so the picture is not one lucky example.

In [7]:
DEMO_QUERIES = [
    "Why was the White Rabbit in such a hurry?",
    "What game does the Queen of Hearts make everyone play?",
    "Who does Alice meet at the mad tea party?",
    "How does the Cheshire Cat disappear?",
    "What happens at the trial of the Knave of Hearts?",
]

print(f"{'query':<48} {'kept':>5} {'new':>4} {'max climb':>10}")
print("-" * 72)

total_new = 0
for q in DEMO_QUERIES:
    cands, rr = hybrid_then_rerank(q, n_candidates=50, top_k=5)
    stage1_top5 = set(cands[:5])
    final = [i for i, _ in rr]
    kept = len(stage1_top5 & set(final))
    new = 5 - kept
    total_new += new
    climbs = [cands.index(i) + 1 - (pos + 1) for pos, i in enumerate(final)]
    print(f"{q[:46]:<48} {kept:>5} {new:>4} {max(climbs):>10}")

print("-" * 72)
print(f"across {len(DEMO_QUERIES)} queries, {total_new} of "
      f"{5 * len(DEMO_QUERIES)} final slots were filled by documents "
      f"outside the stage-1 top 5")

query                                             kept  new  max climb
------------------------------------------------------------------------


Why was the White Rabbit in such a hurry?            3    2         27


What game does the Queen of Hearts make everyo       3    2         27


Who does Alice meet at the mad tea party?            4    1         13


How does the Cheshire Cat disappear?                 5    0          2


What happens at the trial of the Knave of Hear       5    0          1
------------------------------------------------------------------------
across 5 queries, 5 of 25 final slots were filled by documents outside the stage-1 top 5


"max climb" is the number of positions the biggest mover gained. A climb of 20
means a paragraph that RRF buried at rank 25 - completely invisible to a
`k=5` retriever - turned out to be one of the five best documents in the corpus
for that question.

That is the concrete benefit of a wide candidate set: **the reranker can only
promote what stage 1 retrieved.** Widening from 5 to 50 candidates costs one
extra vector scan; failing to widen makes the reranker useless.

### 6. End-to-end answers

Now generate with Groq from both context sets, on the same question, so you can
read the difference rather than infer it from ranks.

In [8]:
def generate(doc_ids, query):
    context = "\n\n".join(f"[doc_{i}] {paragraphs[i]}" for i in doc_ids)
    prompt = (
        "Answer the question using ONLY the context below. Be concrete and "
        "quote the detail you used. If the context does not contain the answer, "
        "say exactly that.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )
    return ask(prompt)


q = "How does the Cheshire Cat disappear?"
cands, rr = hybrid_then_rerank(q, n_candidates=50, top_k=3)

print("Q:", q)
print("\n[A] hybrid RRF top 3 -> docs", cands[:3])
print(generate(cands[:3], q))

Q: How does the Cheshire Cat disappear?

[A] hybrid RRF top 3 -> docs [154, 160, 122]


The context does not contain the answer.


In [9]:
print("[B] hybrid + rerank top 3 -> docs", [i for i, _ in rr])
print(generate([i for i, _ in rr], q))

[B] hybrid + rerank top 3 -> docs [154, 164, 160]


Based on [doc_164], the Cheshire Cat disappears by fading away: "The Cat’s head began fading away the moment he was gone, and, by the time he had come back with the Duchess, it had entirely disappeared".


### 7. Design notes and pitfalls

- **Widen stage 1 when you add a reranker.** The classic mistake is to keep
  `k=5` and bolt a reranker on top: it reorders five documents and changes
  almost nothing. Retrieve 30-100.
- **Keep RRF.** It is nearly free and it improves the *candidate pool* by
  merging two different notions of relevance. Better candidates in means better
  reranking out.
- **The reranker replaces the RRF score, it does not blend with it.** Mixing
  them (`0.3 * rrf + 0.7 * ce`) is possible but the scales are incompatible and
  it rarely helps. If you want stage-1 signal to survive, express it by adding
  more candidates, not by blending.
- **Dedupe before reranking.** Near-duplicate chunks waste both candidate slots
  and cross-encoder time, and they crowd the final context with one repeated
  fact.
- **Log both rankings.** Storing stage-1 and stage-2 positions is how you debug
  a retrieval regression later - it tells you immediately which stage failed.

### 8. Key takeaways

- Hybrid RRF and cross-encoder reranking are **complementary**: RRF fuses
  positions across retrievers, the reranker reads text against the query.
- The full stack is BM25 + dense -> RRF (50 candidates) -> cross-encoder (top 5)
  -> LLM.
- A wide candidate set is what makes reranking worth anything; documents that
  climb 15-25 positions are the whole point.
- The reranker cannot rescue a document stage 1 never retrieved.

Next: [`04_measuring_the_lift.ipynb`](04_measuring_the_lift.ipynb) stops
eyeballing and measures precision@k before and after, on a fixed question set.